#Initialization

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab

Mounted at /content/drive
/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab


In [2]:
import os
os.environ["PYTHONHASHSEED"] = "123"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be before torch import

In [3]:
import math, random, hashlib, copy
import pandas as pd
import numpy as np

from pathlib import Path
from typing  import Tuple, List
from PIL     import Image, ImageEnhance

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from sklearn.metrics import fbeta_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

#Data Loading

In [4]:
train_sc1a_df = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1a/train_sc1a.xlsx")
val_sc1a_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1a/val_sc1a.xlsx")
test_sc1a_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1a/test_sc1a.xlsx")

train_sc1b_df = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1b/train_sc1b.xlsx")
val_sc1b_df   = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1b/val_sc1b.xlsx")
test_sc1b_df  = pd.read_excel("/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/train-val-test/sc1b/test_sc1b.xlsx")

In [5]:
# Split Summarize Data

def split_hash(ids):
    txt = "\n".join(sorted(map(str, ids)))
    return hashlib.sha256(txt.encode("utf-8")).hexdigest()

def summarize_split(df, name):
    child_labels = (
        df.groupby("child_id")["label"]
          .agg(lambda s: int(pd.Series.mode(s).iloc[0]))
          .value_counts()
          .sort_index()
          .to_dict()
    )
    label_counts = df["label"].value_counts().sort_index().to_dict()
    return {
        "split": name,
        "n_images": int(len(df)),
        "n_children": int(df["child_id"].nunique()),
        "child_label_0": int(child_labels.get(0, 0)),
        "child_label_1": int(child_labels.get(1, 0)),
        "image_label_0": int(label_counts.get(0, 0)),
        "image_label_1": int(label_counts.get(1, 0)),
        "child_id_hash": split_hash(df["child_id"].unique().tolist()),
    }

##Summary Sc.1.a. Dataset

In [6]:
split_summary = pd.DataFrame([
    summarize_split(train_sc1a_df, "train"),
    summarize_split(val_sc1a_df, "val"),
    summarize_split(test_sc1a_df, "test"),
])

print("=== SUBJECT-DISJOINT SPLIT SUMMARY ===")
display(split_summary.iloc[:,:-1])

=== SUBJECT-DISJOINT SPLIT SUMMARY ===


,split,n_images,n_children,child_label_0,child_label_1,image_label_0,image_label_1
0,train,3200,100,50,50,1600,1600
1,val,400,99,49,50,200,200
2,test,400,100,50,50,200,200


##Summary Sc.1.b. Dataset

In [7]:
split_summary = pd.DataFrame([
    summarize_split(train_sc1b_df, "train"),
    summarize_split(val_sc1b_df, "val"),
    summarize_split(test_sc1b_df, "test"),
])

print("=== SUBJECT-DISJOINT SPLIT SUMMARY ===")
display(split_summary.iloc[:,:-1])

=== SUBJECT-DISJOINT SPLIT SUMMARY ===


,split,n_images,n_children,child_label_0,child_label_1,image_label_0,image_label_1
0,train,3200,80,40,40,1600,1600
1,val,400,10,5,5,200,200
2,test,400,10,5,5,200,200


#Pre-processing Set

In [8]:
SEED = 123
EPOCHS = 20

In [9]:
# Transformation

IMAGE_SIZE = 224

imagenet_norm = transforms.Normalize([0.485, 0.456, 0.406],
                                     [0.229, 0.224, 0.225])

train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    imagenet_norm,
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    imagenet_norm,
])

In [10]:
# Dataset Loader

BATCH_SIZE = 32
NUM_WORKERS = 2

class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx]
        image  = Image.open(sample["path"]).convert("RGB")
        image  = self.transform(image) if self.transform else transforms.ToTensor()(image)
        label  = int(sample["label"])
        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "child_id": str(sample["child_id"]),
            "path": str(sample["path"]),
        }

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(dataframe, transform, shuffle):
    ds = FaceDataset(dataframe, transform=transform)
    g  = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        generator=g,
        worker_init_fn=seed_worker,  # ← ADD THIS
    )

#EfficientNet-B0

In [11]:
sc1_train_result = []

In [12]:
# Model - pre-trained EfficientNet B0

def create_model(num_classes=2):
    weights = EfficientNet_B0_Weights.DEFAULT
    model   = efficientnet_b0(weights=weights)
    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)
    return model

In [13]:
# Modeling: Train per-epoch Function

def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(mode=train_mode)

    total_loss, total_correct, total_count = 0.0, 0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, labels)

        if train_mode:
            loss.backward()
            optimizer.step()

        preds = logits.argmax(1)
        total_loss += loss.item() * images.size(0)
        total_correct += (preds == labels).sum().item()
        total_count += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    avg_loss = total_loss / max(1, total_count)
    avg_acc  = total_correct / max(1, total_count)
    f2       = fbeta_score(all_labels, all_preds, beta=2, pos_label=1, zero_division=0)

    tn = ((all_preds == 0) & (all_labels == 0)).sum()
    fp = ((all_preds == 1) & (all_labels == 0)).sum()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return avg_loss, avg_acc, f2, specificity

#Sc.1.a. Model Development

##Sc.1.a. EfficientNet-B0 with SGD

###Sc.1.a. EfficientNet-B0 with SGD CE Loss Function

In [14]:
criterion = nn.CrossEntropyLoss(weight=None)

In [15]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)

Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 112MB/s] 


###Sc.1.a. SGD Optimizer

In [16]:
# SGD
LR          = 1e-3
MOMENTUM    = 0.9
WEIGHT_DECAY= 1e-4
NESTEROV    = True

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=NESTEROV
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.1.a. EfficientNet-B0 with SGD Training

In [17]:
# Load Dataset

dl_train_sc1a = make_loader(train_sc1a_df, train_tfms, shuffle=True)
dl_val_sc1a   = make_loader(val_sc1a_df, eval_tfms, shuffle=False)
dl_test_sc1a  = make_loader(test_sc1a_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc1a_df)} | val={len(val_sc1a_df)} | test={len(test_sc1a_df)}")


train=3200 | val=400 | test=400


In [18]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1a_sgd.pth"

In [19]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc1a, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc1a, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc1_train_result.append([
    "sc1a-efficientnet-b0-sgd",
    "sc1a", "efficientnet-b0", "sgd",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.001
    maximize: False
    momentum: 0.9
    nesterov: True
    weight_decay: 0.0001
)
Loss Function : CrossEntropyLoss()
Loss Weight   : None
Epoch 01/20 | train loss 0.5912 acc 0.7334 f2 0.7407 spec 0.7231 | val loss 0.4354 acc 0.9200 f2 0.9137 spec 0.9300
Epoch 02/20 | train loss 0.3387 acc 0.9256 f2 0.9149 spec 0.9425 | val loss 0.1705 acc 0.9925 f2 0.9940 spec 0.9900
Epoch 03/20 | train loss 0.1527 acc 0.9819 f2 0.9804 spec 0.9844 | val loss 0.0573 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 04/20 | train loss 0.0777 acc 0.9922 f2 0.9928 spec 0.9912 | val loss 0.0238 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 05/20 | train loss 0.0452 acc 0.9953 f2 0.9947 spec 0.9962 | val loss 0.0126 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 06/20 | train loss 0.0317 acc 0.9975 f2 0.9967 spec 0.9988 | val loss 0.0086 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 07/20

##Sc.1.a. EfficientNet-B0 with Adam

###Sc.1.a. EfficientNet-B0 with Adam CE Loss Function

In [20]:
criterion = nn.CrossEntropyLoss(weight=None)

In [21]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)

Device: cuda


###Sc.1.a. Adam Optimizer

In [22]:
# Adam
LR          = 1e-4
BETAS       = (0.9, 0.999)
EPS         = 1e-8
WEIGHT_DECAY= 0.0

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.1.a. EfficientNet-B0 with Adam Training

In [23]:
# Load Dataset

dl_train_sc1a = make_loader(train_sc1a_df, train_tfms, shuffle=True)
dl_val_sc1a   = make_loader(val_sc1a_df, eval_tfms, shuffle=False)
dl_test_sc1a  = make_loader(test_sc1a_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc1a_df)} | val={len(val_sc1a_df)} | test={len(test_sc1a_df)}")


train=3200 | val=400 | test=400


In [24]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1a_adam.pth"

In [25]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc1a, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc1a, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc1_train_result.append([
    "sc1a-efficientnet-b0-adam",
    "sc1a", "efficientnet-b0", "adam",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.0
)
Loss Function : CrossEntropyLoss()
Loss Weight   : None
Epoch 01/20 | train loss 0.2821 acc 0.9094 f2 0.9118 spec 0.9056 | val loss 0.0125 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 02/20 | train loss 0.0257 acc 0.9956 f2 0.9968 spec 0.9938 | val loss 0.0021 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 03/20 | train loss 0.0120 acc 0.9984 f2 0.9986 spec 0.9981 | val loss 0.0009 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 04/20 | train loss 0.0088 acc 0.9984 f2 0.9990 spec 0.9975 | val loss 0.0018 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 05/20 | train loss 0.0077 acc 0.9984 f2 0.9986 spec 0.9981 | val loss 0.0004 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 06/20 | train loss 0.0035 acc 0.9997 f2 0.9995 spec 1.0000 | 

##Sc.1.a. EfficientNet-B0 with AdamW

###Sc.1.a. EfficientNet-B0 with AdamW CE Loss Function

In [26]:
criterion = nn.CrossEntropyLoss(weight=None)

In [27]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)


Device: cuda


###Sc.1.a. AdamW Optimizer

In [28]:
# Optimizer Parameter Set

LR = 1e-4
BETAS = (0.9, 0.999)
EPS = 1e-8
WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.1.a. EfficientNet-B0 with AdamW Training

In [29]:
# Load Dataset

dl_train_sc1a = make_loader(train_sc1a_df, train_tfms, shuffle=True)
dl_val_sc1a   = make_loader(val_sc1a_df, eval_tfms, shuffle=False)
dl_test_sc1a  = make_loader(test_sc1a_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc1a_df)} | val={len(val_sc1a_df)} | test={len(test_sc1a_df)}")


train=3200 | val=400 | test=400


In [30]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1a_adamw.pth"

In [31]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc1a, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc1a, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc1_train_result.append([
    "sc1a-efficientnet-b0-adamw",
    "sc1a", "efficientnet-b0", "adamw",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
)
Loss Function : CrossEntropyLoss()
Loss Weight   : None
Epoch 01/20 | train loss 0.2822 acc 0.9094 f2 0.9118 spec 0.9056 | val loss 0.0125 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 02/20 | train loss 0.0257 acc 0.9956 f2 0.9968 spec 0.9938 | val loss 0.0021 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 03/20 | train loss 0.0120 acc 0.9984 f2 0.9986 spec 0.9981 | val loss 0.0009 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 04/20 | train loss 0.0088 acc 0.9984 f2 0.9990 spec 0.9975 | val loss 0.0018 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 05/20 | train loss 0.0077 acc 0.9984 f2 0.9986 spec 0.9981 | val loss 0.0004 acc 1.0000 f2 1.0000 spec 1.0000
Epoch 06/20 | train loss 0.0035 acc 0.9997 f2 0.9995 spec 1.0000 |

#Sc.1.b. Model Development

##Sc.1.b. EfficientNet-B0 with SGD

###Sc.1.b. EfficientNet-B0 with SGD CE Loss Function

In [32]:
criterion = nn.CrossEntropyLoss(weight=None)

In [33]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)

Device: cuda


###Sc.1.b. SGD Optimizer

In [34]:
# SGD
LR          = 1e-3
MOMENTUM    = 0.9
WEIGHT_DECAY= 1e-4
NESTEROV    = True

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=NESTEROV
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.1.b. EfficientNet-B0 with SGD Training

In [35]:
# Load Dataset

dl_train_sc1b = make_loader(train_sc1b_df, train_tfms, shuffle=True)
dl_val_sc1b   = make_loader(val_sc1b_df, eval_tfms, shuffle=False)
dl_test_sc1b  = make_loader(test_sc1b_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc1b_df)} | val={len(val_sc1b_df)} | test={len(test_sc1b_df)}")


train=3200 | val=400 | test=400


In [36]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1b_sgd.pth"

In [37]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc1b, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc1b, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc1_train_result.append([
    "sc1b-efficientnet-b0-sgd",
    "sc1b", "efficientnet-b0", "sgd",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.001
    lr: 0.001
    maximize: False
    momentum: 0.9
    nesterov: True
    weight_decay: 0.0001
)
Loss Function : CrossEntropyLoss()
Loss Weight   : None
Epoch 01/20 | train loss 0.5731 acc 0.7631 f2 0.7812 spec 0.7369 | val loss 0.6316 acc 0.6500 f2 0.7448 spec 0.5150
Epoch 02/20 | train loss 0.2860 acc 0.9594 f2 0.9598 spec 0.9587 | val loss 0.6867 acc 0.5525 f2 0.7247 spec 0.3100
Epoch 03/20 | train loss 0.1148 acc 0.9881 f2 0.9904 spec 0.9844 | val loss 0.7251 acc 0.5750 f2 0.7532 spec 0.3200
Epoch 04/20 | train loss 0.0556 acc 0.9956 f2 0.9964 spec 0.9944 | val loss 0.8238 acc 0.5575 f2 0.7293 spec 0.3150
Epoch 05/20 | train loss 0.0359 acc 0.9981 f2 0.9977 spec 0.9988 | val loss 0.8800 acc 0.5600 f2 0.7332 spec 0.3150
Epoch 06/20 | train loss 0.0279 acc 0.9978 f2 0.9969 spec 0.9994 | val loss 0.9845 acc 0.5875 f2 0.7909 spec 0.2900
Epoch 07/20

##Sc.1.b. EfficintNet-B0 with Adam

###Sc.1.b. EfficientNet-B0 with Adam CE Loss Function

In [38]:
criterion = nn.CrossEntropyLoss(weight=None)

In [39]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)

Device: cuda


###Sc.1.b. Adam Optimizer

In [40]:
# Adam
LR          = 1e-4
BETAS       = (0.9, 0.999)
EPS         = 1e-8
WEIGHT_DECAY= 0.0

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.1.b. EfficientNet-B0 with Adam Training

In [41]:
# Load Dataset

dl_train_sc1b = make_loader(train_sc1b_df, train_tfms, shuffle=True)
dl_val_sc1b   = make_loader(val_sc1b_df, eval_tfms, shuffle=False)
dl_test_sc1b  = make_loader(test_sc1b_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc1b_df)} | val={len(val_sc1b_df)} | test={len(test_sc1b_df)}")


train=3200 | val=400 | test=400


In [42]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1b_adam.pth"

In [43]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc1b, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc1b, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc1_train_result.append([
    "sc1b-efficientnet-b0-adam",
    "sc1b", "efficientnet-b0", "adam",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.0
)
Loss Function : CrossEntropyLoss()
Loss Weight   : None
Epoch 01/20 | train loss 0.2477 acc 0.9319 f2 0.9385 spec 0.9213 | val loss 0.9010 acc 0.6200 f2 0.8063 spec 0.3450
Epoch 02/20 | train loss 0.0182 acc 0.9978 f2 0.9976 spec 0.9981 | val loss 1.0883 acc 0.6150 f2 0.7675 spec 0.3950
Epoch 03/20 | train loss 0.0088 acc 0.9988 f2 0.9991 spec 0.9981 | val loss 1.2982 acc 0.6275 f2 0.7774 spec 0.4100
Epoch 04/20 | train loss 0.0046 acc 0.9997 f2 0.9995 spec 1.0000 | val loss 1.3724 acc 0.6425 f2 0.7360 spec 0.5100
Epoch 05/20 | train loss 0.0040 acc 0.9994 f2 0.9994 spec 0.9994 | val loss 1.0704 acc 0.6025 f2 0.7479 spec 0.3950
Epoch 06/20 | train loss 0.0069 acc 0.9972 f2 0.9962 spec 0.9988 | 

##Sc.1.b. EfficientNet-B0 with AdamW

###Sc.1.b. EfficientNet-B0 with AdamW CE Loss Function

In [44]:
criterion = nn.CrossEntropyLoss(weight=None)

In [45]:
# Model Initialization

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Random Configuration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = create_model(num_classes=2).to(device)

Device: cuda


###Sc.1.b. AdamW Optimizer

In [46]:
# Optimizer Parameter Set

LR = 1e-4
BETAS = (0.9, 0.999)
EPS = 1e-8
WEIGHT_DECAY = 1e-2

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    betas=BETAS,
    eps=EPS,
    weight_decay=WEIGHT_DECAY
    )

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

###Sc.1.b. EfficientNet-B0 with AdamW Training

In [47]:
# Load Dataset

dl_train_sc1b = make_loader(train_sc1b_df, train_tfms, shuffle=True)
dl_val_sc1b   = make_loader(val_sc1b_df, eval_tfms, shuffle=False)
dl_test_sc1b  = make_loader(test_sc1b_df, eval_tfms, shuffle=False)

print(f"\ntrain={len(train_sc1b_df)} | val={len(val_sc1b_df)} | test={len(test_sc1b_df)}")


train=3200 | val=400 | test=400


In [48]:
base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/model/"
BEST_PATH = base + "best_model_efficientnet_b0_sc1b_adamw.pth"

In [49]:
# Training Loop

print("Optimizer     :", optimizer)
print("Loss Function :", criterion)
print("Loss Weight   :", criterion.weight)

# ---------------------------------------------------------
# Best-model tracking
# ---------------------------------------------------------

best_val_f2   = -1.0
best_val_loss = float("inf")
best_epoch    = None
best_state    = None

best_val_f2_constrained   = -1.0
best_val_f2_unconstrained = -1.0
best_val_loss_constrained   = float("inf")
best_val_loss_unconstrained = float("inf")
best_state_constrained    = None
best_state_unconstrained  = None
best_epoch_constrained    = None
best_epoch_unconstrained  = None

MIN_SPECIFICITY = 0.30
MIN_DELTA = 1e-4

# ---------------------------------------------------------
# Training
# ---------------------------------------------------------

for epoch in range(1, EPOCHS + 1):

    tr_loss, tr_acc, tr_f2, tr_spec = run_epoch(model, dl_train_sc1b, criterion, optimizer=optimizer)
    val_loss, val_acc, val_f2, val_spec = run_epoch(model, dl_val_sc1b, criterion, optimizer=None)
    scheduler.step()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {tr_loss:.4f} acc {tr_acc:.4f} "
        f"f2 {tr_f2:.4f} spec {tr_spec:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} "
        f"f2 {val_f2:.4f} spec {val_spec:.4f}"
    )

    meets_specificity = val_spec >= MIN_SPECIFICITY

    is_better_unconstrained = (
        val_f2 > best_val_f2_unconstrained + MIN_DELTA
    ) or (
        np.isclose(val_f2, best_val_f2_unconstrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_unconstrained
    )

    if is_better_unconstrained:
        best_val_f2_unconstrained = val_f2
        best_val_loss_unconstrained = val_loss
        best_epoch_unconstrained = epoch
        best_state_unconstrained = copy.deepcopy(model.state_dict())

    is_better_constrained = meets_specificity and (
        val_f2 > best_val_f2_constrained + MIN_DELTA
    ) or (
        meets_specificity
        and np.isclose(val_f2, best_val_f2_constrained, atol=MIN_DELTA)
        and val_loss < best_val_loss_constrained
    )

    if is_better_constrained:
        best_val_f2_constrained = val_f2
        best_val_loss_constrained = val_loss
        best_epoch_constrained = epoch
        best_state_constrained = copy.deepcopy(model.state_dict())

    if best_state_constrained is not None:
        best_state = best_state_constrained
        best_epoch = best_epoch_constrained
        best_val_f2 = best_val_f2_constrained
        best_val_loss = best_val_loss_constrained
        selection_mode = "constrained (specificity met)"
    else:
        best_state = best_state_unconstrained
        best_epoch = best_epoch_unconstrained
        best_val_f2 = best_val_f2_unconstrained
        best_val_loss = best_val_loss_unconstrained

        selection_mode = ("unconstrained (fallback — no epoch met specificity)")


if best_state is not None:
    #model.load_state_dict(best_state)
    torch.save(best_state,BEST_PATH)

    print("\nModel selected via:", selection_mode)
    print(f"Best epoch: {best_epoch} | val_f2: {best_val_f2:.4f} | val_loss: {best_val_loss:.4f}")
else:
    raise RuntimeError("No valid model state was recorded.")

sc1_train_result.append([
    "sc1b-efficientnet-b0-adamw",
    "sc1b", "efficientnet-b0", "adamw",
    selection_mode,
    best_epoch,
    best_val_f2,
    ])

Optimizer     : AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 0.0001
    maximize: False
    weight_decay: 0.01
)
Loss Function : CrossEntropyLoss()
Loss Weight   : None
Epoch 01/20 | train loss 0.2477 acc 0.9319 f2 0.9385 spec 0.9213 | val loss 0.9012 acc 0.6200 f2 0.8063 spec 0.3450
Epoch 02/20 | train loss 0.0183 acc 0.9978 f2 0.9976 spec 0.9981 | val loss 1.0886 acc 0.6150 f2 0.7675 spec 0.3950
Epoch 03/20 | train loss 0.0088 acc 0.9988 f2 0.9991 spec 0.9981 | val loss 1.2985 acc 0.6275 f2 0.7774 spec 0.4100
Epoch 04/20 | train loss 0.0046 acc 0.9997 f2 0.9995 spec 1.0000 | val loss 1.3726 acc 0.6425 f2 0.7360 spec 0.5100
Epoch 05/20 | train loss 0.0040 acc 0.9994 f2 0.9994 spec 0.9994 | val loss 1.0700 acc 0.6025 f2 0.7479 spec 0.3950
Epoch 06/20 | train loss 0.0069 acc 0.9972 f2 0.9962 spec 0.9988 |

#Result

In [50]:
sc1_train_result_df = pd.DataFrame(sc1_train_result,
                                   columns=['model_name',
                                            'scheme', 'model', 'optimizer',
                                            'selection_mode',
                                            'best_epoch',
                                            'best_val_f2',
                                            ])
display(sc1_train_result_df)

base = "/content/drive/MyDrive/Stunted Children Identification/stunting-identification-lab/result/"
sc1_train_result_df.to_excel(base + 'sc1_train_result.xlsx', index=False)

,model_name,scheme,model,optimizer,selection_mode,best_epoch,best_val_f2
0,sc1a-efficientnet-b0-sgd,sc1a,efficientnet-b0,sgd,constrained (specificity met),19,1.000000
1,sc1a-efficientnet-b0-adam,sc1a,efficientnet-b0,adam,constrained (specificity met),19,1.000000
2,sc1a-efficientnet-b0-adamw,sc1a,efficientnet-b0,adamw,constrained (specificity met),19,1.000000
3,sc1b-efficientnet-b0-sgd,sc1b,efficientnet-b0,sgd,constrained (specificity met),7,0.792301
4,sc1b-efficientnet-b0-adam,sc1b,efficientnet-b0,adam,constrained (specificity met),10,0.840708
5,sc1b-efficientnet-b0-adamw,sc1b,efficientnet-b0,adamw,constrained (specificity met),10,0.840708
